# 02 — Random Forest grouped validation

This notebook reads the pixel-level CSV exported by
`01_GEE_feature_sampling.js`, performs spatially grouped validation,
compares predefined feature stacks, tunes the selected Random Forest,
and saves a reproducible final model.

Methodological safeguards:

- The source CSV is read only.
- Folds are assigned at polygon level using `group_uid`, never by pixel.
- Every pixel from the same spatial group remains in one fold.
- Each polygon contributes at most 100 pixels by default.
- Training weights make every retained polygon contribute equal total weight.
- Metrics are reported at weighted-pixel and polygon levels.
- Feature-stack and hyperparameter selection are repeated inside each
  outer training split for an honest nested grouped estimate.
- Longitude, latitude, `valid_count`, IDs, and group fields are never predictors.

## 重要：请以 notebook 方式打开

不要把 `.ipynb` 文件的原始文本粘贴到 Python 单元运行。`.ipynb` 本质上是
JSON 文件，其中的 `"execution_count": null` 是合法 notebook 元数据，但
`null` 不是 Python 变量，所以粘贴运行会出现 `NameError`。

正确方式：在 Colab 选择 **File → Open notebook → Upload**，上传整个
`02_RF_grouped_validation.ipynb`。打开后会看到下面这些相互独立的代码单元，
从上到下逐个运行即可。运行途中可修改配置单元，但修改后应从相关步骤重新运行。

### 0.1 挂载 Google Drive

- **作用：** 让 Colab 能访问你的训练 CSV 和结果目录。
- **输入：** Google 账号授权。
- **输出：** 可访问 `<DURIAN_DATA_ROOT>/`。
- **耗时：** 很短；首次运行需要确认授权。

In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


### 0.2 导入 Python 库

- **作用：** 加载表格处理、绘图、分组验证和 Random Forest 所需库。
- **输入：** Colab 默认 Python 环境。
- **输出：** 后续单元可使用 pandas、sklearn、joblib 等。
- **耗时：** 数秒。

In [ ]:
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('once')
sns.set_theme(style='whitegrid', context='notebook')

### 0.3 设置输入与本次结果文件夹

- **作用：** 锁定 Drive CSV，并为本次运行建立独立的时间戳目录。重复运行本单元时沿用当前目录。
- **输入：** Drive 中的 `Bentong_pixel_samples_2025_v1.csv`。
- **输出：** `OUTPUT_DIR` 及 tables、figures、models、metadata 子目录。
- **耗时：** 很短。

In [ ]:
INPUT_CSV = DATA_ROOT / r"Bentong_pixel_samples_2025_v1.csv"
OUTPUT_ROOT = REPO_OUTPUT_ROOT

# Re-running this cell in the same Colab session keeps the current folder.
if 'OUTPUT_DIR' not in globals():
    run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    OUTPUT_DIR = OUTPUT_ROOT / f'RF_grouped_validation_{run_stamp}'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
METADATA_DIR = OUTPUT_DIR / 'metadata'

for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

print('Input CSV:', INPUT_CSV)
print('Output folder:', OUTPUT_DIR)
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

## 1. Locked configuration

These definitions must remain identical to the GEE sampling and later
classification scripts. Change them only before model development begins.

### 1.1 设置随机种子、fold 数和像元上限

- **作用：** 集中放置最常修改的实验参数。
- **输入：** 人工设置。
- **输出：** 可复现的运行参数。
- **耗时：** 立即完成。

In [ ]:
RANDOM_SEED = 42
N_SPLITS = 5
INNER_SPLITS = 3
MAX_PIXELS_PER_SAMPLE = 100
N_JOBS = 2

### 1.2 锁定七类名称与编码

- **作用：** 确保 CSV、模型和后续 GEE 分类使用同一类别映射。
- **输入：** 七类定义。
- **输出：** `CLASS_TO_ID`、`ID_TO_CLASS`、`CLASS_IDS`。
- **耗时：** 立即完成。

In [ ]:
CLASS_TO_ID = {
    'Built-up/Bare soil': 0,
    'Durian': 1,
    'Forest': 2,
    'Mixed agriculture': 3,
    'Oil palm': 4,
    'Rubber': 5,
    'Water': 6,
}
ID_TO_CLASS = {value: key for key, value in CLASS_TO_ID.items()}
CLASS_IDS = sorted(ID_TO_CLASS)
CLASS_NAMES = [ID_TO_CLASS[class_id] for class_id in CLASS_IDS]
DURIAN_ID = CLASS_TO_ID['Durian']

### 1.3 定义 S2、S1、DEM predictors 与特征组合

- **作用：** 明确哪些字段可进入模型，并建立待比较的五套 feature stacks。
- **输入：** GEE 导出字段名。
- **输出：** `FULL_BANDS` 和 `FEATURE_SETS`。
- **耗时：** 立即完成。

In [ ]:
S2_BANDS = [
    'B3', 'B4', 'B5', 'B6', 'B7',
    'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDRE', 'NDWI', 'EVI',
]
S1_BANDS = [
    'VV_median', 'VH_median',
    'VV_stdDev', 'VH_stdDev',
    'VV_p10', 'VV_p90',
    'VH_p10', 'VH_p90',
    'VH_VV_ratio_dB',
]
DEM_BANDS = ['elevation', 'slope', 'eastness', 'northness']
FULL_BANDS = S2_BANDS + S1_BANDS + DEM_BANDS

FEATURE_SETS = {
    'S2': S2_BANDS,
    'S1': S1_BANDS,
    'S2_S1': S2_BANDS + S1_BANDS,
    'S2_DEM': S2_BANDS + DEM_BANDS,
    'FULL': FULL_BANDS,
}

### 1.4 定义必需元数据与 baseline RF

- **作用：** 区分模型 predictors、标识字段和 QA 字段，并设置特征比较的固定 RF。
- **输入：** 抽样 CSV 结构和实验设计。
- **输出：** 字段清单与 `BASELINE_RF_PARAMS`。
- **耗时：** 立即完成。

In [ ]:
REQUIRED_METADATA = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
    'class_id', 'CLUSTER_ID', 'domain', 'longitude', 'latitude',
]
QA_COLUMNS = ['valid_count']

BASELINE_RF_PARAMS = {
    'n_estimators': 300,
    'max_features': 'sqrt',
    'min_samples_leaf': 2,
    'max_depth': None,
    'class_weight': 'balanced_subsample',
    'bootstrap': True,
}

# Deliberately small candidate set for transparent, limited tuning.

### 1.5 定义有限 RF 参数候选

- **作用：** 使用可解释的小型候选集进行调参，避免无边界搜索。
- **输入：** 六组预设参数。
- **输出：** `RF_CANDIDATES`。
- **耗时：** 立即完成。

In [ ]:
RF_CANDIDATES = [
    {
        'candidate_id': 'C0', 'n_estimators': 300,
        'max_features': 'sqrt', 'min_samples_leaf': 1,
        'max_depth': None, 'class_weight': 'balanced_subsample',
        'bootstrap': True,
    },
    {
        'candidate_id': 'C1', 'n_estimators': 500,
        'max_features': 'sqrt', 'min_samples_leaf': 2,
        'max_depth': None, 'class_weight': 'balanced_subsample',
        'bootstrap': True,
    },
    {
        'candidate_id': 'C2', 'n_estimators': 500,
        'max_features': 'sqrt', 'min_samples_leaf': 5,
        'max_depth': None, 'class_weight': 'balanced_subsample',
        'bootstrap': True,
    },
    {
        'candidate_id': 'C3', 'n_estimators': 500,
        'max_features': 0.5, 'min_samples_leaf': 2,
        'max_depth': None, 'class_weight': 'balanced_subsample',
        'bootstrap': True,
    },
    {
        'candidate_id': 'C4', 'n_estimators': 300,
        'max_features': 'sqrt', 'min_samples_leaf': 2,
        'max_depth': 20, 'class_weight': 'balanced_subsample',
        'bootstrap': True,
    },
    {
        'candidate_id': 'C5', 'n_estimators': 500,
        'max_features': 'sqrt', 'min_samples_leaf': 2,
        'max_depth': None, 'class_weight': None,
        'bootstrap': True,
    },
]

print('Locked classes:', CLASS_TO_ID)
print('Full predictor count:', len(FULL_BANDS))
print('Feature sets:', {name: len(bands) for name, bands in FEATURE_SETS.items()})

## 2. Load and audit the raw pixel CSV

The notebook never overwrites the source CSV. Any exclusions are recorded
in the timestamped output folder.

### 2.1 读取 CSV 并核对字段

- **作用：** 只读加载原始像元表，检查缺失和额外字段。
- **输入：** `INPUT_CSV`。
- **输出：** `raw_df` 和字段审计信息。
- **耗时：** 取决于 CSV 大小，通常数秒至数十秒。

In [ ]:
raw_df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f'Raw rows: {len(raw_df):,}')
print(f'Raw columns: {raw_df.shape[1]}')
print(f'Approximate memory: {raw_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

expected_columns = REQUIRED_METADATA + FULL_BANDS + QA_COLUMNS
missing_columns = sorted(set(expected_columns) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {missing_columns}')

unexpected_columns = sorted(set(raw_df.columns) - set(expected_columns))
print('Unexpected extra columns (retained but not modelled):', unexpected_columns)

### 2.2 统一类型并核对类别编码

- **作用：** 转换数值字段、处理 inf，并确认 `class_lv2` 与 `class_id` 一致。
- **输入：** `raw_df`。
- **输出：** 标准化后的 `df`。
- **耗时：** 较短。

In [ ]:
df = raw_df.copy()
df['class_lv2'] = df['class_lv2'].astype(str).str.strip()

numeric_columns = FULL_BANDS + QA_COLUMNS + [
    'class_id', 'CLUSTER_ID', 'longitude', 'latitude'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df[FULL_BANDS] = df[FULL_BANDS].replace([np.inf, -np.inf], np.nan)

expected_class_ids = df['class_lv2'].map(CLASS_TO_ID)
unknown_classes = sorted(df.loc[expected_class_ids.isna(), 'class_lv2'].unique())
if unknown_classes:
    raise ValueError(f'Unknown class labels: {unknown_classes}')

mismatched_class_ids = (
    df['class_id'].notna()
    & (df['class_id'].astype('Int64') != expected_class_ids.astype('Int64'))
)
if mismatched_class_ids.any():
    examples = df.loc[
        mismatched_class_ids,
        ['class_lv2', 'class_id']
    ].drop_duplicates().head(20)
    raise ValueError(f'class_id mapping mismatch:\n{examples}')

# Reapply the locked mapping after validation.
df['class_id'] = expected_class_ids.astype(int)

if df['pixel_uid'].isna().any() or df['pixel_uid'].duplicated().any():
    raise ValueError(
        'pixel_uid must be present and globally unique. '
        f'Null={df["pixel_uid"].isna().sum()}, '
        f'duplicates={df["pixel_uid"].duplicated().sum()}'
    )

### 2.3 记录并排除缺少必要数据的像元

- **作用：** 不修改原始 CSV，只把无法建模的行记录到结果目录后排除。
- **输入：** 标准化 `df`。
- **输出：** 排除日志和清理后的 `df`。
- **耗时：** 较短。

In [ ]:
# Record and remove only rows lacking a predictor or essential identifier.
essential_columns = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'class_id'
] + FULL_BANDS
exclusion_mask = df[essential_columns].isna().any(axis=1)
exclusion_summary = {
    'raw_rows': int(len(df)),
    'excluded_rows_missing_essential_data': int(exclusion_mask.sum()),
    'retained_rows': int((~exclusion_mask).sum()),
}
with open(METADATA_DIR / 'data_exclusion_summary.json', 'w', encoding='utf-8') as file:
    json.dump(exclusion_summary, file, indent=2, ensure_ascii=False)

if exclusion_mask.any():
    df.loc[
        exclusion_mask,
        ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2']
    ].to_csv(TABLE_DIR / 'excluded_rows_identifiers.csv', index=False)

df = df.loc[~exclusion_mask].copy().reset_index(drop=True)

### 2.4 检查 sample、group 与类别关系

- **作用：** 验证一个 polygon 只有一个类别和空间组，一个 group 不混合类别，并核对 557/206。
- **输入：** 清理后的 `df`。
- **输出：** 完整性检查结果；不一致时主动停止。
- **耗时：** 较短。

In [ ]:
# One polygon must have exactly one class and one spatial group.
sample_integrity = df.groupby('sample_uid').agg(
    class_count=('class_id', 'nunique'),
    group_count=('group_uid', 'nunique'),
)
if sample_integrity[['class_count', 'group_count']].to_numpy().max() != 1:
    raise ValueError('At least one sample_uid maps to multiple classes or groups.')

# One group must not mix classes in the current class-specific grouping design.
group_class_counts = df.groupby('group_uid')['class_id'].nunique()
if group_class_counts.max() != 1:
    raise ValueError('At least one group_uid contains multiple classes.')

sample_count = df['sample_uid'].nunique()
group_count = df['group_uid'].nunique()
if sample_count != 557 or group_count != 206:
    raise ValueError(
        f'Expected 557 samples and 206 groups after cleaning; '
        f'found {sample_count} samples and {group_count} groups.'
    )

### 2.5 生成原始数据摘要表

- **作用：** 统计每类像元、polygon、group 以及每个 polygon 的像元量。
- **输入：** 通过完整性检查的 `df`。
- **输出：** `class_summary_raw.csv` 和 `sample_pixel_counts_raw.csv`。
- **耗时：** 较短。

In [ ]:
class_summary_raw = (
    df.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        samples=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
        valid_count_min=('valid_count', 'min'),
        valid_count_median=('valid_count', 'median'),
    )
    .reset_index()
    .sort_values('class_id')
)
sample_pixel_counts_raw = (
    df.groupby(['sample_uid', 'group_uid', 'class_id', 'class_lv2'])
    .size()
    .rename('pixels_raw')
    .reset_index()
)

class_summary_raw.to_csv(TABLE_DIR / 'class_summary_raw.csv', index=False)
sample_pixel_counts_raw.to_csv(
    TABLE_DIR / 'sample_pixel_counts_raw.csv', index=False
)

print(class_summary_raw.to_string(index=False))
display(df.head())

### 2.6 绘制平衡前的像元分布

- **作用：** 直观看出类别与 polygon 面积导致的像元数量差异。
- **输入：** 原始摘要表。
- **输出：** `raw_pixel_distribution.png`。
- **耗时：** 数秒。

In [ ]:
# Visual audit: pixel contribution before balancing.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=class_summary_raw,
    x='class_lv2', y='pixel_rows', ax=axes[0], color='#3B82F6'
)
axes[0].set_title('Raw pixel rows by class')
axes[0].set_xlabel('')
axes[0].set_ylabel('Pixel rows')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(
    data=sample_pixel_counts_raw,
    x='class_lv2', y='pixels_raw', ax=axes[1], color='#A7F3D0'
)
axes[1].set_title('Raw pixels per polygon')
axes[1].set_xlabel('')
axes[1].set_ylabel('Pixels per polygon')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'raw_pixel_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

## 3. Balance polygon contributions

A large polygon otherwise contributes many more training rows than a small
polygon. The notebook caps retained pixels per polygon and assigns a weight
of `1 / retained_pixels` so every polygon has total training weight 1.

### 3.1 对每个 polygon 确定性抽样

- **作用：** 每个 polygon 最多保留 100 个像元，固定种子保证重复运行结果一致。
- **输入：** 完整 `df`。
- **输出：** `model_df`。
- **耗时：** 通常数秒至一分钟。

In [ ]:
def stable_seed(text, base_seed=RANDOM_SEED):
    digest = hashlib.sha256(str(text).encode('utf-8')).hexdigest()
    return (int(digest[:8], 16) + base_seed) % (2**32 - 1)


retained_parts = []
for sample_uid, group in df.groupby('sample_uid', sort=True):
    group = group.sort_values('pixel_uid')
    if len(group) > MAX_PIXELS_PER_SAMPLE:
        group = group.sample(
            n=MAX_PIXELS_PER_SAMPLE,
            random_state=stable_seed(sample_uid),
            replace=False,
        )
    retained_parts.append(group)

model_df = (
    pd.concat(retained_parts, ignore_index=True)
    .sort_values('pixel_uid')
    .reset_index(drop=True)
)

### 3.2 计算等 polygon 总权重并保存摘要

- **作用：** 使每个 polygon 的样本权重总和为 1，避免大 polygon 主导训练。
- **输入：** 抽样后的 `model_df`。
- **输出：** `sample_weight` 与 `sample_pixel_counts_model.csv`。
- **耗时：** 较短。

In [ ]:
retained_counts = model_df.groupby('sample_uid')['pixel_uid'].transform('size')
model_df['sample_weight'] = 1.0 / retained_counts

polygon_total_weights = model_df.groupby('sample_uid')['sample_weight'].sum()
if not np.allclose(polygon_total_weights.to_numpy(), 1.0):
    raise AssertionError('Polygon training weights do not sum to 1.')

sample_pixel_counts_model = (
    model_df.groupby(['sample_uid', 'group_uid', 'class_id', 'class_lv2'])
    .size()
    .rename('pixels_retained')
    .reset_index()
    .merge(sample_pixel_counts_raw, how='left')
)
sample_pixel_counts_model['retention_fraction'] = (
    sample_pixel_counts_model['pixels_retained']
    / sample_pixel_counts_model['pixels_raw']
)
sample_pixel_counts_model.to_csv(
    TABLE_DIR / 'sample_pixel_counts_model.csv', index=False
)

print(f'Rows before cap: {len(df):,}')
print(f'Rows used for modelling: {len(model_df):,}')
print(f'Maximum retained pixels per polygon: {MAX_PIXELS_PER_SAMPLE}')
print('Polygon total-weight range:', polygon_total_weights.min(), polygon_total_weights.max())

## 4. Create fixed polygon-level grouped folds

Fold allocation is created on one row per polygon, then mapped back to all
retained pixels. These folds are reused for every feature configuration.

### 4.1 建立 polygon 级分组表并检查 fold 可行性

- **作用：** 每个 polygon 只保留一行，用于按 `group_uid` 分配 folds。
- **输入：** `model_df`。
- **输出：** `sample_table` 和每类 group 数量。
- **耗时：** 较短。

In [ ]:
sample_table = (
    model_df[['sample_uid', 'group_uid', 'class_id', 'class_lv2']]
    .drop_duplicates('sample_uid')
    .sort_values('sample_uid')
    .reset_index(drop=True)
)

groups_per_class = (
    sample_table.groupby(['class_id', 'class_lv2'])['group_uid']
    .nunique()
    .rename('groups')
    .reset_index()
)
if groups_per_class['groups'].min() < N_SPLITS:
    raise ValueError(
        f'{N_SPLITS}-fold CV is not feasible; the minimum groups in a class is '
        f'{groups_per_class["groups"].min()}.'
    )

### 4.2 分配固定的五个 grouped folds

- **作用：** 同一 `group_uid` 的所有 polygon 永远进入同一验证 fold。
- **输入：** polygon 级 `sample_table`。
- **输出：** 每个 `sample_uid` 的 `fold_id`。
- **耗时：** 较短。

In [ ]:
def find_valid_grouped_folds(
    sample_table,
    n_splits,
    class_ids,
    base_seed=42,
    max_attempts=1000,
):
    """寻找每个 fold 均包含全部类别、且分布相对平衡的分组方案。"""

    dummy_x = np.zeros((len(sample_table), 1))

    best_fold_ids = None
    best_seed = None
    best_balance_score = np.inf

    for attempt in range(max_attempts):
        seed = base_seed + attempt

        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )

        fold_ids = np.full(len(sample_table), -1, dtype=int)

        for fold_id, (_, validation_indices) in enumerate(
            splitter.split(
                dummy_x,
                sample_table["class_id"],
                groups=sample_table["group_uid"],
            )
        ):
            fold_ids[validation_indices] = fold_id

        test_table = sample_table.copy()
        test_table["fold_id"] = fold_ids

        # 每个 fold 中各类别的 polygon 数量
        sample_counts = pd.crosstab(
            test_table["fold_id"],
            test_table["class_id"],
        ).reindex(
            index=range(n_splits),
            columns=class_ids,
            fill_value=0,
        )

        # 跳过缺少任意类别的方案
        if (sample_counts == 0).any().any():
            continue

        # 同时考虑各类别 polygon 和 group 的平衡程度
        group_counts = (
            test_table.groupby(["fold_id", "class_id"])["group_uid"]
            .nunique()
            .unstack(fill_value=0)
            .reindex(
                index=range(n_splits),
                columns=class_ids,
                fill_value=0,
            )
        )

        sample_proportions = sample_counts.div(
            sample_counts.sum(axis=0),
            axis=1,
        )
        group_proportions = group_counts.div(
            group_counts.sum(axis=0),
            axis=1,
        )

        expected = 1.0 / n_splits

        balance_score = (
            ((sample_proportions - expected) ** 2).to_numpy().sum()
            + ((group_proportions - expected) ** 2).to_numpy().sum()
        )

        if balance_score < best_balance_score:
            best_balance_score = balance_score
            best_fold_ids = fold_ids.copy()
            best_seed = seed

    if best_fold_ids is None:
        raise ValueError(
            f"尝试 {max_attempts} 个随机种子后，仍无法让每个 fold 包含全部类别。"
            "请检查 groups_per_class，必要时把 N_SPLITS 从 5 改为 4。"
        )

    return best_fold_ids, best_seed, best_balance_score


fold_ids, selected_fold_seed, fold_balance_score = (
    find_valid_grouped_folds(
        sample_table=sample_table,
        n_splits=N_SPLITS,
        class_ids=CLASS_IDS,
        base_seed=RANDOM_SEED,
        max_attempts=1000,
    )
)

sample_table["fold_id"] = fold_ids

print("Selected fold seed:", selected_fold_seed)
print("Fold balance score:", fold_balance_score)

display(
    pd.crosstab(
        sample_table["fold_id"],
        sample_table["class_lv2"],
    )
)

### 4.3 审计 folds 并映射回像元

- **作用：** 确认 group 不跨 fold、每个 fold 包含七类，并保存 fold 表。
- **输入：** 带 `fold_id` 的 `sample_table`。
- **输出：** 更新后的 `model_df`、fold CSV 表。
- **耗时：** 较短。

In [ ]:
# A group must occur in exactly one validation fold.
if sample_table.groupby('group_uid')['fold_id'].nunique().max() != 1:
    raise AssertionError('A group_uid was assigned to more than one fold.')

fold_class_samples = (
    sample_table.groupby(['fold_id', 'class_id', 'class_lv2'])
    .agg(
        samples=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
)

observed_classes_per_fold = sample_table.groupby('fold_id')['class_id'].nunique()
if observed_classes_per_fold.min() != len(CLASS_IDS):
    raise ValueError('At least one fold does not contain all seven classes.')

model_df = model_df.merge(
    sample_table[['sample_uid', 'fold_id']],
    on='sample_uid',
    how='left',
    validate='many_to_one',
)

sample_table.to_csv(TABLE_DIR / 'fold_assignments.csv', index=False)
fold_class_samples.to_csv(TABLE_DIR / 'fold_class_summary.csv', index=False)
groups_per_class.to_csv(TABLE_DIR / 'groups_per_class.csv', index=False)

display(fold_class_samples)

### 4.4 绘制各 fold 的类别与 group 平衡图

- **作用：** 检查五个验证 folds 的样本和空间组分布。
- **输入：** `fold_class_samples`。
- **输出：** `fold_balance.png`。
- **耗时：** 数秒。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(
    data=fold_class_samples,
    x='fold_id', y='samples', hue='class_lv2', ax=axes[0]
)
axes[0].set_title('Polygon samples by fold and class')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Polygons')

sns.barplot(
    data=fold_class_samples,
    x='fold_id', y='groups', hue='class_lv2', ax=axes[1]
)
axes[1].set_title('Spatial groups by fold and class')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Groups')

for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fold_balance.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Modelling and evaluation helpers

Model selection prioritizes polygon-level Durian F1, then polygon-level
macro F1. Pixel metrics use equal-polygon weights.

### 5.1 定义 RF 训练函数

- **作用：** 统一模型参数、随机种子和 polygon 权重的使用方式。
- **输入：** 训练 dataframe、features、RF 参数。
- **输出：** 已拟合的 RandomForestClassifier。
- **耗时：** 这里只定义函数，不训练。

In [ ]:
def clean_rf_params(params):
    return {key: value for key, value in params.items() if key != 'candidate_id'}


def fit_rf(train_frame, features, params, seed_offset=0):
    model = RandomForestClassifier(
        **clean_rf_params(params),
        random_state=RANDOM_SEED + seed_offset,
        n_jobs=N_JOBS,
    )
    model.fit(
        train_frame[features],
        train_frame['class_id'],
        sample_weight=train_frame['sample_weight'],
    )
    return model

### 5.2 定义像元预测函数

- **作用：** 输出类别预测以及七类概率列。
- **输入：** 模型、验证像元、features。
- **输出：** 像元级预测 dataframe。
- **耗时：** 这里只定义函数。

In [ ]:
def make_pixel_predictions(model, frame, features):
    predicted_ids = model.predict(frame[features]).astype(int)
    probabilities = model.predict_proba(frame[features])

    output_columns = [
        'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
        'class_id', 'fold_id', 'sample_weight'
    ]
    output = frame[output_columns].copy()
    output['pred_class_id'] = predicted_ids
    output['pred_class_name'] = output['pred_class_id'].map(ID_TO_CLASS)

    # Always create one probability column per locked class.
    for class_id in CLASS_IDS:
        output[f'prob_{class_id}'] = 0.0
    for column_index, class_id in enumerate(model.classes_.astype(int)):
        output[f'prob_{class_id}'] = probabilities[:, column_index]

    return output

### 5.3 定义 polygon 概率聚合函数

- **作用：** 对同一 polygon 的像元类别概率取平均，再确定 polygon 类别。
- **输入：** 像元级预测。
- **输出：** polygon 级预测。
- **耗时：** 这里只定义函数。

In [ ]:
def aggregate_polygon_predictions(pixel_predictions):
    probability_columns = [f'prob_{class_id}' for class_id in CLASS_IDS]
    aggregations = {
        'group_uid': 'first',
        'class_lv2': 'first',
        'class_id': 'first',
        'fold_id': 'first',
    }
    aggregations.update({column: 'mean' for column in probability_columns})

    polygon_predictions = (
        pixel_predictions.groupby('sample_uid', as_index=False)
        .agg(aggregations)
    )

    probability_matrix = polygon_predictions[probability_columns].to_numpy()
    class_array = np.asarray(CLASS_IDS)
    polygon_predictions['pred_class_id'] = class_array[
        probability_matrix.argmax(axis=1)
    ]
    polygon_predictions['pred_class_name'] = (
        polygon_predictions['pred_class_id'].map(ID_TO_CLASS)
    )
    return polygon_predictions

### 5.4 定义评价指标与统一评估函数

- **作用：** 计算 accuracy、balanced accuracy、macro F1 与 Durian precision/recall/F1。
- **输入：** 真实类别、预测类别及可选权重。
- **输出：** 像元级和 polygon 级指标。
- **耗时：** 这里只定义函数。

In [ ]:
def metric_dictionary(y_true, y_pred, sample_weight=None):
    precision, recall, durian_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[DURIAN_ID],
        average=None,
        sample_weight=sample_weight,
        zero_division=0,
    )
    return {
        'accuracy': accuracy_score(y_true, y_pred, sample_weight=sample_weight),
        'balanced_accuracy': balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'macro_f1': f1_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            average='macro',
            sample_weight=sample_weight,
            zero_division=0,
        ),
        'durian_precision': float(precision[0]),
        'durian_recall': float(recall[0]),
        'durian_f1': float(durian_f1[0]),
    }


def evaluate_model(model, validation_frame, features):
    pixel_predictions = make_pixel_predictions(model, validation_frame, features)
    polygon_predictions = aggregate_polygon_predictions(pixel_predictions)

    pixel_metrics = metric_dictionary(
        pixel_predictions['class_id'],
        pixel_predictions['pred_class_id'],
        sample_weight=pixel_predictions['sample_weight'],
    )
    polygon_metrics = metric_dictionary(
        polygon_predictions['class_id'],
        polygon_predictions['pred_class_id'],
    )

    combined_metrics = {
        **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
        **{f'polygon_{key}': value for key, value in polygon_metrics.items()},
    }
    return combined_metrics, pixel_predictions, polygon_predictions

### 5.5 定义混淆矩阵和分类报告函数

- **作用：** 统一保存后续精度表和图。
- **输入：** 预测结果。
- **输出：** CSV/PNG 混淆矩阵与报告 dataframe。
- **耗时：** 这里只定义函数。

In [ ]:
def save_confusion_figure(y_true, y_pred, title, output_name, weights=None):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        sample_weight=weights,
    )
    matrix_frame = pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )
    matrix_frame.to_csv(TABLE_DIR / f'{output_name}.csv')

    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix_frame, annot=True, fmt='.1f', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted class')
    plt.ylabel('Reference class')
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f'{output_name}.png', dpi=180, bbox_inches='tight')
    plt.show()


def report_frame(y_true, y_pred, weights=None):
    report = classification_report(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        sample_weight=weights,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T

## 6. Predefined feature-stack comparison

A fixed baseline RF and the same outer folds are used for every feature
stack. This isolates data-source contribution from hyperparameter changes.

### 6.1 运行五套特征组合的固定 RF 比较

- **作用：** 使用完全相同的 grouped folds 和 baseline RF 比较数据源贡献。
- **输入：** `model_df`、`FEATURE_SETS`、folds。
- **输出：** 每个 feature set × fold 的指标。
- **耗时：** 第一个明显耗时单元，训练 25 个 RF。

In [ ]:
feature_fold_records = []

for feature_set_name, features in FEATURE_SETS.items():
    print(f'Feature comparison: {feature_set_name} ({len(features)} bands)')
    for fold_id in range(N_SPLITS):
        train_frame = model_df.loc[model_df['fold_id'] != fold_id]
        validation_frame = model_df.loc[model_df['fold_id'] == fold_id]

        model = fit_rf(
            train_frame,
            features,
            BASELINE_RF_PARAMS,
            seed_offset=fold_id,
        )
        metrics, _, _ = evaluate_model(model, validation_frame, features)
        feature_fold_records.append({
            'feature_set': feature_set_name,
            'fold_id': fold_id,
            'feature_count': len(features),
            **metrics,
        })

feature_fold_metrics = pd.DataFrame(feature_fold_records)
feature_fold_metrics.to_csv(
    TABLE_DIR / 'feature_set_fold_metrics.csv', index=False
)

### 6.2 汇总特征组合结果并选择完整数据候选

- **作用：** 以 polygon Durian F1 为主、macro F1 为辅排序并绘图。
- **输入：** feature-fold 指标。
- **输出：** `feature_set_summary.csv`、比较图和 `SELECTED_FEATURE_SET`。
- **耗时：** 较短。

In [ ]:
feature_summary = (
    feature_fold_metrics.groupby('feature_set')
    .agg(
        feature_count=('feature_count', 'first'),
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_durian_f1_std=('polygon_durian_f1', 'std'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        polygon_macro_f1_std=('polygon_macro_f1', 'std'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
        polygon_accuracy_mean=('polygon_accuracy', 'mean'),
    )
    .reset_index()
    .sort_values(
        ['polygon_durian_f1_mean', 'polygon_macro_f1_mean', 'pixel_durian_f1_mean'],
        ascending=False,
    )
    .reset_index(drop=True)
)
feature_summary.to_csv(TABLE_DIR / 'feature_set_summary.csv', index=False)

SELECTED_FEATURE_SET = feature_summary.loc[0, 'feature_set']
SELECTED_FEATURES = FEATURE_SETS[SELECTED_FEATURE_SET]

print('Selected feature set:', SELECTED_FEATURE_SET)
display(feature_summary)

plot_data = feature_summary.melt(
    id_vars=['feature_set'],
    value_vars=['polygon_durian_f1_mean', 'polygon_macro_f1_mean'],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x='feature_set', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('Grouped feature-stack comparison (polygon level)')
plt.xlabel('Feature set')
plt.ylabel('Mean 5-fold score')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'feature_set_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Nested grouped feature selection and RF tuning

For each outer fold, the feature stack is selected with the baseline RF
using only the outer training data. Candidate RF settings are then compared
on that selected stack, again inside the outer training data. The resulting
pipeline is evaluated once on the untouched outer validation fold.

### 7.1 定义内层 grouped CV 评分函数

- **作用：** 只在一个外层训练集内部评价 feature stack 或 RF 参数。
- **输入：** 外层训练像元与 polygon 表。
- **输出：** 内层三折平均指标。
- **耗时：** 这里只定义函数。

In [ ]:
def inner_configuration_score(
    outer_train_pixels,
    outer_train_samples,
    features,
    rf_params,
    outer_fold_id,
):
    inner_splitter = StratifiedGroupKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED + 1000 + outer_fold_id,
    )
    inner_samples = outer_train_samples.reset_index(drop=True)
    inner_dummy_x = np.zeros((len(inner_samples), 1))
    fold_records = []

    for inner_fold_id, (train_indices, validation_indices) in enumerate(
        inner_splitter.split(
            inner_dummy_x,
            inner_samples['class_id'],
            groups=inner_samples['group_uid'],
        )
    ):
        train_uids = set(inner_samples.loc[train_indices, 'sample_uid'])
        validation_uids = set(
            inner_samples.loc[validation_indices, 'sample_uid']
        )

        train_frame = outer_train_pixels.loc[
            outer_train_pixels['sample_uid'].isin(train_uids)
        ]
        validation_frame = outer_train_pixels.loc[
            outer_train_pixels['sample_uid'].isin(validation_uids)
        ]

        model = fit_rf(
            train_frame,
            features,
            rf_params,
            seed_offset=(outer_fold_id + 1) * 100 + inner_fold_id,
        )
        metrics, _, _ = evaluate_model(model, validation_frame, features)
        fold_records.append(metrics)

    fold_frame = pd.DataFrame(fold_records)
    return {
        'polygon_durian_f1_mean': fold_frame['polygon_durian_f1'].mean(),
        'polygon_macro_f1_mean': fold_frame['polygon_macro_f1'].mean(),
        'pixel_durian_f1_mean': fold_frame['pixel_durian_f1'].mean(),
        'pixel_macro_f1_mean': fold_frame['pixel_macro_f1'].mean(),
    }

### 7.2 运行 nested grouped validation

- **作用：** 每个外层 fold 内先选 feature stack，再选 RF 参数，最后仅在未见外层 fold 评价一次。
- **输入：** 固定外层 folds、五套特征组合、六组 RF 候选。
- **输出：** 外层 OOF 像元与 polygon 预测记录。
- **耗时：** 全 notebook 最耗时；请等待单元完成，不要重复启动。

In [ ]:
nested_feature_records = []
nested_candidate_records = []
nested_outer_records = []
nested_selected_candidates = []
oof_pixel_frames = []
oof_polygon_frames = []

nested_start = time.time()
for outer_fold_id in range(N_SPLITS):
    print(f'Nested outer fold {outer_fold_id + 1}/{N_SPLITS}')
    outer_train_pixels = model_df.loc[model_df['fold_id'] != outer_fold_id]
    outer_validation_pixels = model_df.loc[model_df['fold_id'] == outer_fold_id]
    outer_train_samples = sample_table.loc[
        sample_table['fold_id'] != outer_fold_id
    ]

    # Stage A: select the feature stack without touching the outer fold.
    fold_feature_rows = []
    for feature_set_name, features in FEATURE_SETS.items():
        scores = inner_configuration_score(
            outer_train_pixels,
            outer_train_samples,
            features,
            BASELINE_RF_PARAMS,
            outer_fold_id,
        )
        row = {
            'outer_fold_id': outer_fold_id,
            'feature_set': feature_set_name,
            'feature_count': len(features),
            **scores,
        }
        nested_feature_records.append(row)
        fold_feature_rows.append(row)

    fold_feature_frame = (
        pd.DataFrame(fold_feature_rows)
        .sort_values(
            [
                'polygon_durian_f1_mean',
                'polygon_macro_f1_mean',
                'pixel_durian_f1_mean',
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )
    fold_feature_set = fold_feature_frame.loc[0, 'feature_set']
    fold_features = FEATURE_SETS[fold_feature_set]

    # Stage B: tune RF settings on the fold-specific selected stack.
    fold_candidate_rows = []
    for candidate in RF_CANDIDATES:
        scores = inner_configuration_score(
            outer_train_pixels,
            outer_train_samples,
            fold_features,
            candidate,
            outer_fold_id,
        )
        row = {
            'outer_fold_id': outer_fold_id,
            'selected_feature_set': fold_feature_set,
            'candidate_id': candidate['candidate_id'],
            **scores,
        }
        nested_candidate_records.append(row)
        fold_candidate_rows.append(row)

    fold_candidate_frame = (
        pd.DataFrame(fold_candidate_rows)
        .sort_values(
            [
                'polygon_durian_f1_mean',
                'polygon_macro_f1_mean',
                'pixel_durian_f1_mean',
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )
    best_candidate_id = fold_candidate_frame.loc[0, 'candidate_id']
    best_candidate = next(
        candidate for candidate in RF_CANDIDATES
        if candidate['candidate_id'] == best_candidate_id
    )
    nested_selected_candidates.append({
        'outer_fold_id': outer_fold_id,
        'selected_feature_set': fold_feature_set,
        'selected_feature_count': len(fold_features),
        **best_candidate,
    })

    outer_model = fit_rf(
        outer_train_pixels,
        fold_features,
        best_candidate,
        seed_offset=5000 + outer_fold_id,
    )
    metrics, pixel_predictions, polygon_predictions = evaluate_model(
        outer_model,
        outer_validation_pixels,
        fold_features,
    )
    nested_outer_records.append({
        'outer_fold_id': outer_fold_id,
        'selected_feature_set': fold_feature_set,
        'candidate_id': best_candidate_id,
        **metrics,
    })
    oof_pixel_frames.append(pixel_predictions)
    oof_polygon_frames.append(polygon_predictions)

### 7.3 整理并保存 nested CV 中间结果

- **作用：** 保存内层 feature/参数得分、外层指标和 OOF 预测。
- **输入：** 上一单元的记录列表。
- **输出：** nested CSV 表与 OOF prediction CSV。
- **耗时：** 取决于 OOF 表大小，通常较短。

In [ ]:
# ============================================================
# 7.3 低内存保存 nested grouped validation 结果
# ============================================================

import gc
import os

print(
    f"Nested validation runtime: "
    f"{(time.time() - nested_start) / 60:.1f} minutes"
)


# ------------------------------------------------------------
# 1. 整理体积较小的指标表
# ------------------------------------------------------------

nested_feature_scores = pd.DataFrame.from_records(
    nested_feature_records
)

nested_candidate_scores = pd.DataFrame.from_records(
    nested_candidate_records
)

nested_fold_metrics = pd.DataFrame.from_records(
    nested_outer_records
)

selected_candidates_by_fold = pd.DataFrame.from_records(
    nested_selected_candidates
)


nested_feature_scores.to_csv(
    TABLE_DIR / "nested_inner_feature_set_scores.csv",
    index=False,
)

nested_candidate_scores.to_csv(
    TABLE_DIR / "nested_inner_candidate_scores.csv",
    index=False,
)

nested_fold_metrics.to_csv(
    TABLE_DIR / "nested_outer_fold_metrics.csv",
    index=False,
)

selected_candidates_by_fold.to_csv(
    TABLE_DIR / "nested_selected_candidates.csv",
    index=False,
)


# ------------------------------------------------------------
# 2. 逐个 fold 写入 CSV
#
# 不再执行：
# pd.concat(oof_pixel_frames)
#
# 这样不会同时在内存中保留：
# 旧 dataframe + 合并后的 dataframe + CSV 转换缓冲区
# ------------------------------------------------------------

def stream_fold_frames_to_csv(frames, final_path):
    """
    将多个 fold dataframe 依次写入同一个 CSV。

    特点：
    1. 不使用 pd.concat；
    2. 每写完一个 fold 就释放对应 dataframe；
    3. 先写临时文件，全部成功后再替换正式文件；
    4. 如果本单元已经成功运行，可直接复用现有文件。
    """

    final_path = Path(final_path)
    temporary_path = final_path.with_suffix(
        final_path.suffix + ".tmp"
    )

    # 如果 frames 已被释放，并且正式文件存在，直接复用。
    if len(frames) == 0:
        if final_path.exists():
            print(f"Reusing existing file: {final_path}")
            return

        raise RuntimeError(
            f"No prediction frames are available and file does not exist: "
            f"{final_path}"
        )

    # 删除上次中断留下的不完整临时文件。
    if temporary_path.exists():
        temporary_path.unlink()

    total_rows = 0

    try:
        for frame_index in range(len(frames)):
            frame = frames[frame_index]

            if frame is None:
                continue

            write_mode = "w" if total_rows == 0 else "a"
            write_header = total_rows == 0

            frame.to_csv(
                temporary_path,
                mode=write_mode,
                header=write_header,
                index=False,
            )

            total_rows += len(frame)

            print(
                f"Saved fold frame {frame_index + 1}/{len(frames)}: "
                f"{len(frame):,} rows"
            )

            # 立即释放已经写入 Drive 的 fold dataframe。
            frames[frame_index] = None
            del frame
            gc.collect()

        if total_rows == 0:
            raise RuntimeError(
                f"No rows were written to {temporary_path}"
            )

        # 同一目录内原子替换，避免留下不完整的正式文件。
        os.replace(temporary_path, final_path)

        print(
            f"Completed: {final_path.name}, "
            f"{total_rows:,} rows"
        )

    except Exception:
        # 保留正式旧文件，但删除本次未完成的临时文件。
        if temporary_path.exists():
            temporary_path.unlink()

        raise


pixel_prediction_path = (
    TABLE_DIR / "oof_pixel_predictions.csv"
)

polygon_prediction_path = (
    TABLE_DIR / "oof_polygon_predictions.csv"
)


stream_fold_frames_to_csv(
    frames=oof_pixel_frames,
    final_path=pixel_prediction_path,
)

stream_fold_frames_to_csv(
    frames=oof_polygon_frames,
    final_path=polygon_prediction_path,
)


# 清空已经被写入 Drive 的列表。
oof_pixel_frames.clear()
oof_polygon_frames.clear()

gc.collect()


# ------------------------------------------------------------
# 3. 只读取后续评价需要的字段
#
# 完整概率仍保存在 CSV 中，但不再全部载入内存。
# 后面的 7.4 和 7.5 单元可以保持不变。
# ------------------------------------------------------------

oof_pixel_predictions = pd.read_csv(
    pixel_prediction_path,
    usecols=[
        "class_id",
        "pred_class_id",
        "sample_weight",
    ],
    dtype={
        "class_id": "int16",
        "pred_class_id": "int16",
        "sample_weight": "float32",
    },
)

oof_polygon_predictions = pd.read_csv(
    polygon_prediction_path,
    usecols=[
        "class_id",
        "pred_class_id",
    ],
    dtype={
        "class_id": "int16",
        "pred_class_id": "int16",
    },
)


print()
print("Low-memory OOF data loaded successfully.")
print(
    f"Pixel predictions used for metrics: "
    f"{len(oof_pixel_predictions):,}"
)
print(
    f"Polygon predictions used for metrics: "
    f"{len(oof_polygon_predictions):,}"
)

print()
print("Output files:")
print(pixel_prediction_path)
print(polygon_prediction_path)

### 7.4 计算 nested OOF 总体指标

- **作用：** 把所有外层未见样本合并，得到完整 grouped validation 指标。
- **输入：** OOF 像元和 polygon 预测。
- **输出：** `nested_overall_metrics.json`。
- **耗时：** 较短。

In [ ]:
overall_pixel_metrics = metric_dictionary(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    sample_weight=oof_pixel_predictions['sample_weight'],
)
overall_polygon_metrics = metric_dictionary(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
nested_feature_set_counts = {
    str(name): int(count)
    for name, count in selected_candidates_by_fold[
        'selected_feature_set'
    ].value_counts().items()
}
nested_overall_metrics = {
    'full_data_selected_feature_set': SELECTED_FEATURE_SET,
    'outer_fold_selected_feature_set_counts': nested_feature_set_counts,
    **{f'pixel_{key}': value for key, value in overall_pixel_metrics.items()},
    **{f'polygon_{key}': value for key, value in overall_polygon_metrics.items()},
}
with open(
    METADATA_DIR / 'nested_overall_metrics.json', 'w', encoding='utf-8'
) as file:
    json.dump(nested_overall_metrics, file, indent=2, ensure_ascii=False)

### 7.5 输出分类报告与混淆矩阵

- **作用：** 保存七类明细，并分别展示等 polygon 权重像元级和 polygon 级结果。
- **输入：** OOF 预测。
- **输出：** classification reports 与两套 confusion matrices。
- **耗时：** 数秒至数十秒。

In [ ]:
pixel_report = report_frame(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    weights=oof_pixel_predictions['sample_weight'],
)
polygon_report = report_frame(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
pixel_report.to_csv(TABLE_DIR / 'oof_pixel_classification_report.csv')
polygon_report.to_csv(TABLE_DIR / 'oof_polygon_classification_report.csv')

save_confusion_figure(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    'Nested grouped CV — weighted pixel confusion matrix',
    'oof_pixel_confusion_matrix',
    weights=oof_pixel_predictions['sample_weight'],
)
save_confusion_figure(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
    'Nested grouped CV — polygon confusion matrix',
    'oof_polygon_confusion_matrix',
)

print(json.dumps(nested_overall_metrics, indent=2))
display(nested_fold_metrics)

## 8. Select final parameters and fit the full Bentong RF

The external Pahang assessment remains untouched. Therefore all Bentong
groups may be used to select the final limited RF setting and train the
deployable model after internal performance has been estimated by nested CV.

### 8.1 在全部 Bentong folds 上选择最终 RF 参数

- **作用：** 在 nested CV 已完成无偏性能估计后，为部署模型确定最终候选参数。
- **输入：** 完整 Bentong `model_df` 和 `SELECTED_FEATURES`。
- **输出：** `FINAL_RF_PARAMS` 与 candidate score 表。
- **耗时：** 较耗时，训练 30 个 RF。

In [ ]:
final_candidate_records = []
for candidate in RF_CANDIDATES:
    candidate_fold_records = []
    for fold_id in range(N_SPLITS):
        train_frame = model_df.loc[model_df['fold_id'] != fold_id]
        validation_frame = model_df.loc[model_df['fold_id'] == fold_id]
        model = fit_rf(
            train_frame,
            SELECTED_FEATURES,
            candidate,
            seed_offset=9000 + fold_id,
        )
        metrics, _, _ = evaluate_model(
            model, validation_frame, SELECTED_FEATURES
        )
        candidate_fold_records.append(metrics)

    candidate_fold_frame = pd.DataFrame(candidate_fold_records)
    final_candidate_records.append({
        'candidate_id': candidate['candidate_id'],
        'polygon_durian_f1_mean': candidate_fold_frame[
            'polygon_durian_f1'
        ].mean(),
        'polygon_macro_f1_mean': candidate_fold_frame[
            'polygon_macro_f1'
        ].mean(),
        'pixel_durian_f1_mean': candidate_fold_frame[
            'pixel_durian_f1'
        ].mean(),
        'pixel_macro_f1_mean': candidate_fold_frame[
            'pixel_macro_f1'
        ].mean(),
    })

final_candidate_scores = (
    pd.DataFrame(final_candidate_records)
    .sort_values(
        [
            'polygon_durian_f1_mean',
            'polygon_macro_f1_mean',
            'pixel_durian_f1_mean',
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)
final_candidate_scores.to_csv(
    TABLE_DIR / 'final_candidate_scores.csv', index=False
)

FINAL_CANDIDATE_ID = final_candidate_scores.loc[0, 'candidate_id']
FINAL_RF_PARAMS = next(
    candidate for candidate in RF_CANDIDATES
    if candidate['candidate_id'] == FINAL_CANDIDATE_ID
)

### 8.2 使用全部 Bentong 样本训练并保存最终模型

- **作用：** 拟合供下一阶段 GEE 转换使用的 deployable RF。
- **输入：** 最终 features、参数和全部训练像元。
- **输出：** `rf_final_model.joblib` 与 `rf_final_bundle.joblib`。
- **耗时：** 中等。

In [ ]:
final_model = fit_rf(
    model_df,
    SELECTED_FEATURES,
    FINAL_RF_PARAMS,
    seed_offset=10000,
)

joblib.dump(final_model, MODEL_DIR / 'rf_final_model.joblib')
model_bundle = {
    'model': final_model,
    'feature_set': SELECTED_FEATURE_SET,
    'predictor_bands': SELECTED_FEATURES,
    'class_to_id': CLASS_TO_ID,
    'id_to_class': ID_TO_CLASS,
    'rf_params': clean_rf_params(FINAL_RF_PARAMS),
    'candidate_id': FINAL_CANDIDATE_ID,
    'random_seed': RANDOM_SEED,
    'max_pixels_per_sample': MAX_PIXELS_PER_SAMPLE,
}
joblib.dump(model_bundle, MODEL_DIR / 'rf_final_bundle.joblib')

### 8.3 保存最终模型的特征重要性

- **作用：** 输出可用于解释和论文整理的 feature importance。
- **输入：** `final_model`。
- **输出：** feature importance CSV 和 PNG。
- **耗时：** 较短。

In [ ]:
feature_importance = (
    pd.DataFrame({
        'feature': SELECTED_FEATURES,
        'importance': final_model.feature_importances_,
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
feature_importance.to_csv(TABLE_DIR / 'feature_importance.csv', index=False)

plt.figure(figsize=(9, max(5, len(feature_importance) * 0.28)))
sns.barplot(
    data=feature_importance,
    y='feature', x='importance', color='#10B981'
)
plt.title('Final Random Forest feature importance')
plt.xlabel('Mean decrease in impurity')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()

### 8.4 保存训练行清单与复现 manifest

- **作用：** 记录输入哈希、软件版本、features、参数、样本量和 nested 指标。
- **输入：** 本次完整运行状态。
- **输出：** `training_rows_used.csv` 和 `model_manifest.json`。
- **耗时：** 读取 CSV 计算 SHA256 时可能需要数秒。

In [ ]:
# Save the exact rows used to fit the deployable RF.
model_df[[
    'pixel_uid', 'sample_uid', 'group_uid', 'class_id',
    'class_lv2', 'fold_id', 'sample_weight'
]].to_csv(TABLE_DIR / 'training_rows_used.csv', index=False)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': sha256_file(INPUT_CSV),
    'output_directory': str(OUTPUT_DIR),
    'raw_rows': int(len(raw_df)),
    'clean_rows': int(len(df)),
    'model_rows': int(len(model_df)),
    'sample_count': int(model_df['sample_uid'].nunique()),
    'group_count': int(model_df['group_uid'].nunique()),
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'max_pixels_per_sample': MAX_PIXELS_PER_SAMPLE,
    'random_seed': RANDOM_SEED,
    'class_to_id': CLASS_TO_ID,
    'feature_sets': FEATURE_SETS,
    'selected_feature_set': SELECTED_FEATURE_SET,
    'selected_predictor_bands': SELECTED_FEATURES,
    'final_candidate_id': FINAL_CANDIDATE_ID,
    'final_rf_params': clean_rf_params(FINAL_RF_PARAMS),
    'nested_overall_metrics': nested_overall_metrics,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
with open(METADATA_DIR / 'model_manifest.json', 'w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

### 8.5 查看最终选择结果

- **作用：** 在 notebook 中显示最终 feature stack、候选参数与重要性。
- **输入：** 最终模型结果。
- **输出：** 屏幕摘要。
- **耗时：** 立即完成。

In [ ]:
print('Selected final feature set:', SELECTED_FEATURE_SET)
print('Selected final candidate:', FINAL_CANDIDATE_ID)
print('Final parameters:', clean_rf_params(FINAL_RF_PARAMS))
display(final_candidate_scores)
display(feature_importance)

## 9. Final inventory

The model bundle and manifest are the inputs for
`03_RF_to_GEE.ipynb`. The raw CSV remains untouched.

### 9.1 写入 README 并列出全部结果文件

- **作用：** 建立交付清单，方便下一阶段定位模型、指标和图件。
- **输入：** `OUTPUT_DIR` 中已生成的所有结果。
- **输出：** `README.txt` 和 `output_inventory.csv`。
- **耗时：** 较短。

In [ ]:
readme_text = f'''Random Forest grouped validation results
========================================
Input CSV: {INPUT_CSV}
Selected feature set: {SELECTED_FEATURE_SET}
Selected candidate: {FINAL_CANDIDATE_ID}
Predictor count: {len(SELECTED_FEATURES)}
Samples: {model_df['sample_uid'].nunique()}
Groups: {model_df['group_uid'].nunique()}

Key files
---------
models/rf_final_model.joblib
models/rf_final_bundle.joblib
metadata/model_manifest.json
metadata/nested_overall_metrics.json
tables/fold_assignments.csv
tables/feature_set_summary.csv
tables/nested_inner_feature_set_scores.csv
tables/nested_outer_fold_metrics.csv
tables/oof_pixel_predictions.csv
tables/oof_polygon_predictions.csv
tables/feature_importance.csv
figures/oof_pixel_confusion_matrix.png
figures/oof_polygon_confusion_matrix.png
figures/feature_importance.png
'''
(OUTPUT_DIR / 'README.txt').write_text(readme_text, encoding='utf-8')

inventory = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        inventory.append({
            'relative_path': str(path.relative_to(OUTPUT_DIR)),
            'size_bytes': path.stat().st_size,
        })
inventory_frame = pd.DataFrame(inventory)
inventory_frame.to_csv(TABLE_DIR / 'output_inventory.csv', index=False)

print('Completed successfully.')
print('Results folder:', OUTPUT_DIR)
display(inventory_frame)